# Enrichissement OWID — Consommation de viande per capita

## 1. Imports & configuration

In [15]:
import pandas as pd
import numpy as np
import requests
import io
from pathlib import Path

# Chemins
DATA_RAW         = Path('../data/raw')
FOOD_PRODUCTION  = DATA_RAW / 'Food_Production.csv'
FAO_COMPLET      = DATA_RAW / 'FAO_complet_1961_2023.csv'
OUTPUT_OWID      = DATA_RAW / 'owid_meat_consumption.csv'
CACHE_DIR        = DATA_RAW / 'owid_cache'
CACHE_DIR.mkdir(exist_ok=True)

# URLs OWID — nouveau format (v=1&csvType=full requis depuis 2024)
OWID_URLS = {
    'meat_by_type': (
        'https://ourworldindata.org/grapher/per-capita-meat-consumption-by-type-kilograms-per-year.csv'
        '?v=1&csvType=full&useColumnShortNames=false'
    ),
    'food_supply': (
        'https://ourworldindata.org/grapher/daily-per-capita-supply-of-calories.csv'
        '?v=1&csvType=full&useColumnShortNames=false'
    ),
}

# Correspondance OWID → colonnes Food_Production.csv
OWID_TO_IMPACT = {
    'Beef and Buffalo Meat': 'Beef (beef herd)',
    'Pig Meat'             : 'Pig Meat',
    'Poultry Meat'         : 'Poultry Meat',
    'Sheep and Goat Meat'  : 'Lamb & Mutton',
}

print("✅ Imports OK")
print(f"📂 Cache : {CACHE_DIR.absolute()}")

✅ Imports OK
📂 Cache : c:\Users\Emeline\Documents\_DEV\_Projets_Jedha\_impact_envt_project\notebooks\..\data\raw\owid_cache


---
## 2. Téléchargement des données OWID

In [16]:
# 2.1 — Fonction de téléchargement avec cache
def fetch_owid_csv(url, cache_name, force_download=False):
    """
    Télécharge un CSV OWID avec système de cache local.
    Supprime automatiquement le cache si le fichier est vide (0 ligne de données).
    Retourne un DataFrame.
    """
    cache_path = CACHE_DIR / cache_name

    # Invalider le cache si vide (téléchargement précédent raté)
    if cache_path.exists() and not force_download:
        df_check = pd.read_csv(cache_path)
        if len(df_check) == 0:
            print(f"  ⚠️  Cache vide détecté ({cache_name}) — suppression et re-téléchargement")
            cache_path.unlink()
        else:
            print(f"  📂 Cache : {cache_name} ({cache_path.stat().st_size / 1024:.0f} KB, {len(df_check):,} lignes)")
            return df_check

    print(f"  ⏳ Téléchargement : {url[:80]}...")
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'Accept': 'text/csv,text/plain,*/*',
    }
    resp = requests.get(url, headers=headers, timeout=60)
    resp.raise_for_status()

    df = pd.read_csv(io.StringIO(resp.text))
    df.to_csv(cache_path, index=False)

    if len(df) == 0:
        print(f"  ⚠️  Attention : {len(df)} lignes reçues — vérifier l'URL")
    else:
        print(f"  ✅ {len(df):,} lignes — sauvegardé en cache")
    return df

print("✅ Fonction fetch_owid_csv définie")

✅ Fonction fetch_owid_csv définie


In [17]:
# 2.2 — Téléchargement OWID avec essai de plusieurs slugs
# OWID migre régulièrement ses URLs — on tente plusieurs variantes

OWID_SLUGS = [
    'per-capita-meat-consumption-by-type-kilograms-per-year',
    'per-capita-meat-supply-by-type',
    'meat-supply-per-person',
]

df_meat = pd.DataFrame()
SOURCE_LABEL = None

for slug in OWID_SLUGS:
    url = f'https://ourworldindata.org/grapher/{slug}.csv?v=1&csvType=full&useColumnShortNames=false'
    cache_name = f'owid_{slug}.csv'
    print(f"\n🔎 Essai : {slug}")
    try:
        df_try = fetch_owid_csv(url, cache_name)
        if len(df_try) > 0:
            df_meat = df_try
            SOURCE_LABEL = f'OWID grapher ({slug})'
            print(f"  ✅ {len(df_meat):,} lignes — OK")
            break
        else:
            print(f"  ⚠️  0 lignes reçues — essai suivant")
    except Exception as e:
        print(f"  ❌ Erreur : {e}")

if len(df_meat) == 0:
    print("\n⚠️  Toutes les URLs OWID ont échoué.")
    print("→ Voir cellule 2.3 pour le calcul depuis FAO + World Bank")
    SOURCE_LABEL = 'FAO+WB (calculé)'
else:
    print(f"\n📊 Structure brute :")
    print(f"  Lignes   : {len(df_meat):,}")
    print(f"  Colonnes : {df_meat.columns.tolist()}")

# Détecter dynamiquement les colonnes (OWID change parfois les noms)
def detect_col(df, candidates):
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

COL_COUNTRY = detect_col(df_meat, ['Entity', 'entityName', 'country'])
COL_CODE    = detect_col(df_meat, ['Code', 'entityCode', 'iso_code', 'iso3'])
COL_YEAR    = detect_col(df_meat, ['Year', 'year'])

if len(df_meat) > 0:
    print(f"\n🔎 Colonnes détectées : pays='{COL_COUNTRY}' | code='{COL_CODE}' | année='{COL_YEAR}'")


🔎 Essai : per-capita-meat-consumption-by-type-kilograms-per-year
  ⚠️  Cache vide détecté (owid_per-capita-meat-consumption-by-type-kilograms-per-year.csv) — suppression et re-téléchargement
  ⏳ Téléchargement : https://ourworldindata.org/grapher/per-capita-meat-consumption-by-type-kilograms...
  ⚠️  Attention : 0 lignes reçues — vérifier l'URL
  ⚠️  0 lignes reçues — essai suivant

🔎 Essai : per-capita-meat-supply-by-type
  ⚠️  Cache vide détecté (owid_per-capita-meat-supply-by-type.csv) — suppression et re-téléchargement
  ⏳ Téléchargement : https://ourworldindata.org/grapher/per-capita-meat-supply-by-type.csv?v=1&csvTyp...
  ⚠️  Attention : 0 lignes reçues — vérifier l'URL
  ⚠️  0 lignes reçues — essai suivant

🔎 Essai : meat-supply-per-person
  📂 Cache : owid_meat-supply-per-person.csv (384 KB, 12,817 lignes)
  ✅ 12,817 lignes — OK

📊 Structure brute :
  Lignes   : 12,817
  Colonnes : ['Entity', 'Code', 'Year', 'Per capita consumption of meat']

🔎 Colonnes détectées : pays='Entity

In [18]:
# 2.3 — Fallback : calcul depuis FAO + World Bank (si OWID indisponible)
# OWID publie des données FAO FBS divisées par la population — on reproduit ici
# la même logique avec les fichiers qu'on a déjà.

if SOURCE_LABEL == 'FAO+WB (calculé)':
    print("🔧 Calcul de la consommation per capita depuis FAO + World Bank...")

    # Correspondance produits FAO → catégories viande
    MEATS_FAO = {
        'beef'    : ['Bovine Meat', 'Beef and Veal', 'Meat, cattle', 'Bovine'],
        'pork'    : ['Pigmeat', 'Pig meat', 'Meat, pig', 'Pork'],
        'poultry' : ['Poultry Meat', 'Meat, Poultry', 'Meat of poultry'],
        'sheep'   : ['Mutton & Goat Meat', 'Mutton and Goat Meat', 'Sheep meat',
                     'Meat, sheep', 'Ovine Meat', 'Sheep & Goat Meat'],
    }

    # Charger FAO complet (uniquement élément Food)
    print("  📂 Chargement FAO_complet...")
    df_fao_raw = pd.read_csv(FAO_COMPLET, low_memory=False)
    df_fao_food = df_fao_raw[df_fao_raw['Element'] == 'Food'].copy()

    # Charger population (World Bank) — utilise la version interpolée si disponible
    WB_FILE = DATA_RAW / 'worldbank_socioeco_interp.csv'
    if not WB_FILE.exists():
        WB_FILE = DATA_RAW / 'worldbank_socioeco.csv'
    print(f"  📂 Chargement population ({WB_FILE.name})...")
    df_pop = pd.read_csv(WB_FILE, usecols=['iso3', 'nom_fao', 'year', 'population'])
    df_pop = df_pop.dropna(subset=['population'])

    # Construire mapping iso3 depuis le fichier WB
    iso3_map = df_pop[['nom_fao', 'iso3']].dropna().drop_duplicates('nom_fao').set_index('nom_fao')['iso3']

    rows = []
    for cat, keywords in MEATS_FAO.items():
        # Trouver les items FAO qui correspondent à cette catégorie
        pattern = '|'.join(keywords)
        mask = df_fao_food['Item'].str.contains(pattern, case=False, na=False)
        df_cat = df_fao_food[mask]

        if df_cat.empty:
            print(f"  ⚠️  Aucun item trouvé pour '{cat}' (patterns: {keywords[:2]}...)")
            continue

        # Agréger par pays × année (en cas de doublons dans la nomenclature)
        df_agg = (
            df_cat
            .groupby(['Area', 'Year'], as_index=False)['Value']
            .sum()
            .rename(columns={'Value': f'{cat}_1000t'})
        )
        df_agg['iso3'] = df_agg['Area'].map(iso3_map)
        df_agg['category'] = cat
        rows.append(df_agg)
        print(f"  ✅ {cat:<10} : {df_agg['Area'].nunique()} pays, {df_agg['Year'].nunique()} années")

    # Pivot : une ligne par (Area, Year), une colonne par catégorie viande
    df_pivot = None
    for df_cat_agg in rows:
        cat = df_cat_agg['category'].iloc[0]
        df_cat_wide = df_cat_agg[['Area', 'iso3', 'Year', f'{cat}_1000t']].copy()
        if df_pivot is None:
            df_pivot = df_cat_wide
        else:
            df_pivot = df_pivot.merge(df_cat_wide, on=['Area', 'iso3', 'Year'], how='outer')

    if df_pivot is None:
        raise RuntimeError("Aucun item viande trouvé dans FAO_complet — vérifier les noms de produits")

    # Jointure avec population
    df_pivot = df_pivot.merge(
        df_pop[['iso3', 'year', 'population']].rename(columns={'year': 'Year'}),
        on=['iso3', 'Year'],
        how='left'
    )

    # Calcul per capita (kg/habitant/an)
    # 1000 tonnes × 1_000_000 / population = kg/habitant
    for cat in MEATS_FAO.keys():
        col_t = f'{cat}_1000t'
        col_kg = f'{cat}_kg_per_cap'
        if col_t in df_pivot.columns:
            df_pivot[col_kg] = (df_pivot[col_t] * 1_000_000) / df_pivot['population']

    # Renommer pour être compatible avec la suite du notebook
    df_meat = (
        df_pivot
        .rename(columns={'Area': 'nom_owid', 'Year': 'year'})
        [['iso3', 'nom_owid', 'year'] +
         [f'{c}_kg_per_cap' for c in MEATS_FAO.keys() if f'{c}_kg_per_cap' in df_pivot.columns]]
        .dropna(subset=['iso3'])
    )

    COL_COUNTRY = 'nom_owid'
    COL_CODE    = 'iso3'
    COL_YEAR    = 'year'

    print(f"\n✅ Dataset calculé depuis FAO + World Bank :")
    print(f"  Lignes : {len(df_meat):,}")
    print(f"  Pays   : {df_meat['iso3'].nunique()}")
    print(f"  Années : {df_meat['year'].min()}–{df_meat['year'].max()}")
    print(f"  Source : {SOURCE_LABEL}")
    display(df_meat.head())

else:
    # OWID OK — diagnostic des valeurs manquantes
    print(f"✅ Source : {SOURCE_LABEL}")
    print(f"\n🔍 Valeurs manquantes :")
    print(df_meat.isnull().sum().to_string())
    if COL_CODE and COL_COUNTRY:
        sans_code = df_meat[df_meat[COL_CODE].isna()][COL_COUNTRY].unique()
        print(f"\n📋 {len(sans_code)} entités sans code ISO3 (agrégats)")
        for e in sorted(sans_code)[:10]:
            print(f"  - {e}")

✅ Source : OWID grapher (meat-supply-per-person)

🔍 Valeurs manquantes :
Entity                               0
Code                              1868
Year                                 0
Per capita consumption of meat       0

📋 31 entités sans code ISO3 (agrégats)
  - Africa (FAO)
  - Americas (FAO)
  - Asia (FAO)
  - Belgium-Luxembourg (FAO)
  - Caribbean (FAO)
  - Central America (FAO)
  - Central Asia (FAO)
  - China (FAO)
  - Eastern Africa (FAO)
  - Eastern Asia (FAO)


---
## 3. Nettoyage et filtrage

On garde uniquement les pays individuels (Code ISO3 non-null) et on filtre les colonnes de viande.

In [19]:
# 3.1 — Filtrer les pays individuels et normaliser les noms de colonnes
# Si on vient du fallback FAO+WB, df_meat a déjà iso3/nom_owid/year → pas besoin de renommer

if SOURCE_LABEL == 'FAO+WB (calculé)':
    # Déjà propre depuis la cellule 2.3
    df_meat_pays = df_meat.copy()
    print(f"✅ Source FAO+WB — données déjà normalisées")
else:
    # Source OWID : filtrer les agrégats (pas de code ISO3) et renommer
    if COL_CODE:
        df_meat_pays = df_meat[df_meat[COL_CODE].notna()].copy()
    else:
        print("⚠️  Pas de colonne code ISO3 — impossible de filtrer les agrégats")
        df_meat_pays = df_meat.copy()

    rename_cols = {}
    if COL_COUNTRY and COL_COUNTRY != 'nom_owid': rename_cols[COL_COUNTRY] = 'nom_owid'
    if COL_CODE    and COL_CODE    != 'iso3'    : rename_cols[COL_CODE]    = 'iso3'
    if COL_YEAR    and COL_YEAR    != 'year'    : rename_cols[COL_YEAR]    = 'year'
    df_meat_pays = df_meat_pays.rename(columns=rename_cols)

print(f"✅ Lignes : {len(df_meat_pays):,}")
print(f"   Pays   : {df_meat_pays['iso3'].nunique() if 'iso3' in df_meat_pays.columns else 'N/A'}")
print(f"   Années : {df_meat_pays['year'].min()}–{df_meat_pays['year'].max()}")

✅ Lignes : 10,949
   Pays   : 210
   Années : 1961–2022


In [20]:
# 3.2 — Identifier et renommer les colonnes de viande
# Les noms de colonnes OWID peuvent varier selon la version du dataset
print("📋 Toutes les colonnes disponibles :")
for col in df_meat_pays.columns:
    n_non_null = df_meat_pays[col].notna().sum()
    print(f"  [{n_non_null:>6,} non-null] {col}")

# Identifier les colonnes de viande (tout sauf iso3, nom_owid, year)
cols_meta   = ['iso3', 'nom_owid', 'year']
cols_viande = [c for c in df_meat_pays.columns if c not in cols_meta]
print(f"\n🥩 Colonnes de viande identifiées : {len(cols_viande)}")
print(cols_viande)

📋 Toutes les colonnes disponibles :
  [10,949 non-null] nom_owid
  [10,949 non-null] iso3
  [10,949 non-null] year
  [10,949 non-null] Per capita consumption of meat

🥩 Colonnes de viande identifiées : 1
['Per capita consumption of meat']


In [21]:
# 3.3 — Renommage normalisé des colonnes de viande
# On cherche d'abord les types spécifiques (beef/pork/poultry/sheep)
# Si pas de décomposition disponible, on récupère la colonne totale

def find_col(cols, keywords):
    """Retourne la 1ère colonne contenant tous les mots-clés (insensible à la casse)."""
    kw_lower = [k.lower() for k in keywords]
    for c in cols:
        cl = c.lower()
        if all(k in cl for k in kw_lower):
            return c
    return None

rename_map = {}

# Chercher les types de viande spécifiques
c = find_col(cols_viande, ['beef']) or find_col(cols_viande, ['bovine'])
if c: rename_map[c] = 'beef_kg_per_cap'

c = find_col(cols_viande, ['pig']) or find_col(cols_viande, ['pork'])
if c: rename_map[c] = 'pork_kg_per_cap'

c = find_col(cols_viande, ['poultry'])
if c: rename_map[c] = 'poultry_kg_per_cap'

c = find_col(cols_viande, ['sheep']) or find_col(cols_viande, ['mutton']) or find_col(cols_viande, ['goat'])
if c: rename_map[c] = 'sheep_kg_per_cap'

HAS_TYPE_BREAKDOWN = len(rename_map) >= 2

if not HAS_TYPE_BREAKDOWN:
    # Pas de décomposition par type — chercher une colonne "totale"
    c = (find_col(cols_viande, ['total']) or
         find_col(cols_viande, ['meat']) or
         find_col(cols_viande, ['consumption']) or
         (cols_viande[0] if cols_viande else None))
    if c:
        rename_map[c] = 'total_meat_kg_per_cap'
        print(f"⚠️  Pas de décomposition par type de viande")
        print(f"   '{c}' → 'total_meat_kg_per_cap' (colonne totale uniquement)")
        print(f"   → Le calcul CO2 utilisera un facteur moyen pondéré")
else:
    print("✅ Décomposition par type trouvée :")
    for ancien, nouveau in rename_map.items():
        print(f"  '{ancien}' → '{nouveau}'")

non_mappes = [c for c in cols_viande if c not in rename_map]
if non_mappes:
    print(f"\n⚠️  Colonnes non mappées : {non_mappes}")

df_meat_clean = df_meat_pays.rename(columns=rename_map).copy()
print(f"\n✅ Colonnes finales : {df_meat_clean.columns.tolist()}")

⚠️  Pas de décomposition par type de viande
   'Per capita consumption of meat' → 'total_meat_kg_per_cap' (colonne totale uniquement)
   → Le calcul CO2 utilisera un facteur moyen pondéré

✅ Colonnes finales : ['nom_owid', 'iso3', 'year', 'total_meat_kg_per_cap']


---
## 4. Calcul de l'impact carbone per capita

On croise la consommation per capita (OWID) avec les émissions CO₂ par kg (Food_Production.csv)  
pour obtenir l'**empreinte carbone alimentaire per capita** par type de viande.

> `co2_per_capita_kg = conso_kg_per_cap × co2_per_kg_produit`

In [22]:
# 4.1 — Charger Food_Production.csv et extraire les facteurs d'émission
df_impact = pd.read_csv(FOOD_PRODUCTION)
df_impact.columns = df_impact.columns.str.strip()

# Renommer la colonne produit et la colonne CO2 total
col_produit = df_impact.columns[0]   # 'Food product'
col_co2     = 'Total_emissions'       # kgCO2eq/kg

# Extraire les facteurs pour les viandes
VIANDES_IMPACT = {
    'beef_kg_per_cap'    : 'Beef (beef herd)',
    'pork_kg_per_cap'    : 'Pig Meat',
    'poultry_kg_per_cap' : 'Poultry Meat',
    'sheep_kg_per_cap'   : 'Lamb & Mutton',
}

facteurs_co2 = {}
print("🌡️  Facteurs d'émission (kgCO₂eq/kg) :")
for col_owid, nom_impact in VIANDES_IMPACT.items():
    row = df_impact[df_impact[col_produit] == nom_impact]
    if not row.empty:
        co2 = row[col_co2].values[0]
        facteurs_co2[col_owid] = co2
        print(f"  {col_owid:<25} → {nom_impact:<20} : {co2:.1f} kgCO₂eq/kg")
    else:
        print(f"  ⚠️  '{nom_impact}' non trouvé dans Food_Production.csv")

🌡️  Facteurs d'émission (kgCO₂eq/kg) :
  beef_kg_per_cap           → Beef (beef herd)     : 59.6 kgCO₂eq/kg
  pork_kg_per_cap           → Pig Meat             : 7.2 kgCO₂eq/kg
  poultry_kg_per_cap        → Poultry Meat         : 6.1 kgCO₂eq/kg
  sheep_kg_per_cap          → Lamb & Mutton        : 24.5 kgCO₂eq/kg


In [23]:
# 4.2 — Calculer le CO2 per capita par type de viande
df_owid = df_meat_clean.copy()

co2_cols = []

if HAS_TYPE_BREAKDOWN:
    # Cas normal : décomposition par type disponible
    for col_owid, co2_per_kg in facteurs_co2.items():
        if col_owid in df_owid.columns:
            col_co2 = col_owid.replace('_kg_per_cap', '_co2_per_cap')
            df_owid[col_co2] = df_owid[col_owid] * co2_per_kg
            co2_cols.append(col_co2)

    if co2_cols:
        df_owid['total_meat_co2_per_cap'] = df_owid[co2_cols].sum(axis=1, min_count=1)
        print(f"✅ {len(co2_cols)} colonnes CO2 calculées + total")

    # Total conso
    conso_cols = [c for c in df_owid.columns if c.endswith('_kg_per_cap') and c != 'total_meat_kg_per_cap']
    if conso_cols:
        df_owid['total_meat_kg_per_cap'] = df_owid[conso_cols].sum(axis=1, min_count=1)

else:
    # Cas total uniquement : on a seulement total_meat_kg_per_cap
    # On calcule le CO2 avec un facteur moyen pondéré (mix mondial ~13 kgCO2/kg)
    # Pondération approximative : 30% bœuf×60 + 35% porc×7 + 30% volaille×6 + 5% mouton×24 ≈ 25 kgCO2/kg
    # Source : calcul depuis Food_Production.csv avec parts production mondiale FAOSTAT
    if facteurs_co2:
        # Facteur moyen non pondéré sur les 4 types disponibles
        facteur_moyen = sum(facteurs_co2.values()) / len(facteurs_co2)
        print(f"⚠️  Colonne totale uniquement — facteur CO2 moyen utilisé : {facteur_moyen:.1f} kgCO₂eq/kg")
        print(f"   (moyenne non pondérée : beef={facteurs_co2.get('beef_kg_per_cap', '?'):.1f}, "
              f"pork={facteurs_co2.get('pork_kg_per_cap', '?'):.1f}, "
              f"poultry={facteurs_co2.get('poultry_kg_per_cap', '?'):.1f}, "
              f"sheep={facteurs_co2.get('sheep_kg_per_cap', '?'):.1f})")
        df_owid['total_meat_co2_per_cap'] = df_owid['total_meat_kg_per_cap'] * facteur_moyen
        print(f"✅ total_meat_co2_per_cap calculé avec facteur moyen {facteur_moyen:.1f}")
    else:
        print("⚠️  facteurs_co2 vide — CO2 non calculé (relancer la cellule 4.1 d'abord)")

print(f"\n📊 Colonnes calculées :")
for c in [c for c in df_owid.columns if 'co2' in c or 'kg_per_cap' in c]:
    non_null = df_owid[c].notna().sum()
    print(f"  {c:<35} {non_null:,} non-null")

df_owid.head()

⚠️  Colonne totale uniquement — facteur CO2 moyen utilisé : 24.3 kgCO₂eq/kg
   (moyenne non pondérée : beef=59.6, pork=7.2, poultry=6.1, sheep=24.5)
✅ total_meat_co2_per_cap calculé avec facteur moyen 24.3

📊 Colonnes calculées :
  total_meat_kg_per_cap               10,949 non-null
  total_meat_co2_per_cap              10,949 non-null


,nom_owid,iso3,year,total_meat_kg_per_cap,total_meat_co2_per_cap
0,Afghanistan,AFG,1961,14.042126,341.925768
1,Afghanistan,AFG,1962,14.056524,342.276359
2,Afghanistan,AFG,1963,14.470231,352.350125
3,Afghanistan,AFG,1964,14.659532,356.959604
4,Afghanistan,AFG,1965,14.964693,364.390275


---
## 5. Alignement avec les pays FAO

In [24]:
# 5.1 — Vérifier la couverture pays OWID vs FAO
print("📂 Chargement liste pays FAO...")
df_fao = pd.read_csv(FAO_COMPLET, usecols=['Area', 'Area Code'], low_memory=False)
pays_fao = df_fao[['Area', 'Area Code']].drop_duplicates().sort_values('Area')

iso3_owid = set(df_owid['iso3'].unique())
print(f"\n🌍 Pays dans OWID : {len(iso3_owid)}")
print(f"🌍 Pays dans FAO  : {len(pays_fao)}")
print(f"✅ Pays OWID avec données (ont un code ISO3) : {len(iso3_owid)}")
print("\n💡 La jointure avec FAO_complet se fera via la table Dim_Pays (iso3 ↔ pays_fao)")
print("   → Prévu dans le notebook 02_etl_pipeline.ipynb")

📂 Chargement liste pays FAO...

🌍 Pays dans OWID : 210
🌍 Pays dans FAO  : 174
✅ Pays OWID avec données (ont un code ISO3) : 210

💡 La jointure avec FAO_complet se fera via la table Dim_Pays (iso3 ↔ pays_fao)
   → Prévu dans le notebook 02_etl_pipeline.ipynb


In [25]:
# 5.2 — Statistiques descriptives sur quelques pays de référence

# Recalculer les totaux si absents (sécurité)
_conso_cols = [c for c in df_owid.columns if c.endswith('_kg_per_cap') and c != 'total_meat_kg_per_cap']
_co2_cols   = [c for c in df_owid.columns if c.endswith('_co2_per_cap') and c != 'total_meat_co2_per_cap']

if 'total_meat_kg_per_cap' not in df_owid.columns and _conso_cols:
    df_owid['total_meat_kg_per_cap'] = df_owid[_conso_cols].sum(axis=1, min_count=1)

if 'total_meat_co2_per_cap' not in df_owid.columns and _co2_cols:
    df_owid['total_meat_co2_per_cap'] = df_owid[_co2_cols].sum(axis=1, min_count=1)

# Si les co2 cols n'existent pas du tout, les calculer maintenant
if not _co2_cols and 'facteurs_co2' in dir() and facteurs_co2:
    for col_owid, co2_per_kg in facteurs_co2.items():
        if col_owid in df_owid.columns:
            col_c = col_owid.replace('_kg_per_cap', '_co2_per_cap')
            df_owid[col_c] = df_owid[col_owid] * co2_per_kg
    _co2_cols = [c for c in df_owid.columns if c.endswith('_co2_per_cap')]
    if _co2_cols:
        df_owid['total_meat_co2_per_cap'] = df_owid[_co2_cols].sum(axis=1, min_count=1)

# Affichage
PAYS_REF  = ['USA', 'FRA', 'IND', 'CHN', 'BRA', 'ARG', 'ETH']
ANNEES_REF = [1990, 2000, 2010, 2020]

# Adapter les années aux données disponibles
annees_dispo = sorted(df_owid['year'].unique())
ANNEES_REF = [a for a in ANNEES_REF if a in annees_dispo]
if not ANNEES_REF:
    ANNEES_REF = [annees_dispo[i] for i in [-4, -3, -2, -1] if i < len(annees_dispo)]

df_ref = df_owid[
    df_owid['iso3'].isin(PAYS_REF) &
    df_owid['year'].isin(ANNEES_REF)
].copy()

print("🥩 Consommation totale de viande (kg/habitant/an) :")
if 'total_meat_kg_per_cap' in df_ref.columns and not df_ref.empty:
    pivot = df_ref.pivot_table(index='iso3', columns='year', values='total_meat_kg_per_cap', aggfunc='first')
    print(pivot.round(1).to_string())
else:
    print(f"  Colonnes kg disponibles : {_conso_cols}")
    if _conso_cols:
        col = _conso_cols[0]
        print(df_ref.pivot_table(index='iso3', columns='year', values=col, aggfunc='first').round(1).to_string())

print("\n🌡️  Empreinte carbone viande (kgCO₂eq/habitant/an) :")
if 'total_meat_co2_per_cap' in df_ref.columns and not df_ref.empty:
    pivot_co2 = df_ref.pivot_table(index='iso3', columns='year', values='total_meat_co2_per_cap', aggfunc='first')
    print(pivot_co2.round(0).to_string())
else:
    print(f"  Colonnes CO2 disponibles : {_co2_cols}")

🥩 Consommation totale de viande (kg/habitant/an) :
year   1990   2000   2010   2020
iso3                            
ARG    83.4   97.9   95.8  114.4
BRA    49.6   79.2   90.3  100.6
CHN    24.0   44.4   58.3   58.7
ETH     NaN    6.8    8.2    7.1
FRA    98.5   99.6   89.3   83.8
IND     4.1    3.9    4.3    6.2
USA   113.2  123.0  119.3  125.6

🌡️  Empreinte carbone viande (kgCO₂eq/habitant/an) :
year    1990    2000    2010    2020
iso3                                
ARG   2031.0  2384.0  2333.0  2785.0
BRA   1207.0  1929.0  2198.0  2449.0
CHN    583.0  1081.0  1418.0  1430.0
ETH      NaN   166.0   198.0   174.0
FRA   2398.0  2426.0  2173.0  2041.0
IND    101.0    95.0   104.0   151.0
USA   2756.0  2996.0  2904.0  3059.0


---
## 6. Sauvegarde

In [26]:
# 6.1 — Sauvegarder le dataset enrichi
df_owid_final = df_owid.copy()

# Réordonner les colonnes logiquement
cols_id     = ['iso3', 'nom_owid', 'year']
cols_conso  = [c for c in df_owid_final.columns if c.endswith('_kg_per_cap')]
cols_co2    = [c for c in df_owid_final.columns if c.endswith('_co2_per_cap')]
cols_autres = [c for c in df_owid_final.columns if c not in cols_id + cols_conso + cols_co2]

ordre = cols_id + sorted(cols_conso) + sorted(cols_co2) + cols_autres
df_owid_final = df_owid_final[[c for c in ordre if c in df_owid_final.columns]]

df_owid_final.to_csv(OUTPUT_OWID, index=False)

print("💾 Sauvegarde :")
print(f"  ✅ {OUTPUT_OWID.name}")
print(f"     {len(df_owid_final):,} lignes")
print(f"     {df_owid_final['iso3'].nunique()} pays")
print(f"     {df_owid_final['year'].min()}–{df_owid_final['year'].max()}")
print(f"     Colonnes : {df_owid_final.columns.tolist()}")

💾 Sauvegarde :
  ✅ owid_meat_consumption.csv
     10,949 lignes
     210 pays
     1961–2022
     Colonnes : ['iso3', 'nom_owid', 'year', 'total_meat_kg_per_cap', 'total_meat_co2_per_cap']


---
## 7. Validation

In [27]:
# 7.1 — Vérification de cohérence : top pays consommateurs de viande
ANNEE_TEST = 2015

# Recalculer les totaux si absents (exécution partielle possible)
conso_cols_present = [c for c in df_owid_final.columns
                      if c.endswith('_kg_per_cap') and c != 'total_meat_kg_per_cap']
co2_cols_present   = [c for c in df_owid_final.columns
                      if c.endswith('_co2_per_cap') and c != 'total_meat_co2_per_cap']

if 'total_meat_kg_per_cap' not in df_owid_final.columns and conso_cols_present:
    df_owid_final['total_meat_kg_per_cap'] = df_owid_final[conso_cols_present].sum(axis=1, min_count=1)
    print(f"🔧 total_meat_kg_per_cap recalculé depuis : {conso_cols_present}")

if 'total_meat_co2_per_cap' not in df_owid_final.columns and co2_cols_present:
    df_owid_final['total_meat_co2_per_cap'] = df_owid_final[co2_cols_present].sum(axis=1, min_count=1)
    print(f"🔧 total_meat_co2_per_cap recalculé depuis : {co2_cols_present}")

print(f"\n📋 Colonnes disponibles : {df_owid_final.columns.tolist()}")

# Top 10
print(f"\n🏆 Top 10 pays — consommation viande per capita ({ANNEE_TEST}) :")
df_test = df_owid_final[df_owid_final['year'] == ANNEE_TEST]

if df_test.empty:
    annees = sorted(df_owid_final['year'].unique())
    print(f"  ⚠️  Année {ANNEE_TEST} absente — années disponibles : {annees[:5]}...{annees[-3:]}")
    ANNEE_TEST = annees[-5] if len(annees) >= 5 else annees[-1]
    df_test = df_owid_final[df_owid_final['year'] == ANNEE_TEST]
    print(f"  → Utilisation de {ANNEE_TEST} à la place")

cols_affichage = [c for c in ['iso3', 'nom_owid', 'total_meat_kg_per_cap', 'total_meat_co2_per_cap']
                  if c in df_test.columns]

if 'total_meat_kg_per_cap' in df_test.columns:
    top10 = (
        df_test[cols_affichage]
        .dropna(subset=['total_meat_kg_per_cap'])
        .sort_values('total_meat_kg_per_cap', ascending=False)
        .head(10)
    )
    print(top10.to_string(index=False))
    print("\n💡 Attendu : Australie, USA, Argentine, Brésil en haut du classement")
else:
    # Fallback : afficher avec les colonnes individuelles disponibles
    print(f"  Colonnes de consommation : {conso_cols_present}")
    if conso_cols_present:
        col_ref = conso_cols_present[0]
        print(df_test[['iso3', 'nom_owid'] + conso_cols_present]
              .dropna(subset=[col_ref])
              .sort_values(col_ref, ascending=False)
              .head(10)
              .to_string(index=False))


📋 Colonnes disponibles : ['iso3', 'nom_owid', 'year', 'total_meat_kg_per_cap', 'total_meat_co2_per_cap']

🏆 Top 10 pays — consommation viande per capita (2015) :
iso3                         nom_owid  total_meat_kg_per_cap  total_meat_co2_per_cap
 HKG                        Hong Kong             127.747090             3110.641641
 AUS                        Australia             119.777590             2916.584316
 USA                    United States             115.877850             2821.625647
 MAC                            Macao             111.699104             2719.873182
 ARG                        Argentina             110.169846             2682.635750
 WSM                            Samoa             109.152275             2657.857896
 BHS                          Bahamas             107.622970             2620.619319
 ISR                           Israel             105.415820             2566.875217
 PYF                 French Polynesia             102.350220            

In [28]:
# 7.2 — Vérification couverture temporelle
print("📅 Lignes par année (5 ans d'intervalle) :")
cov = df_owid_final.groupby('year').size().reset_index(name='nb_pays')
print(cov[cov['year'] % 5 == 0].to_string(index=False))

print(f"\n📊 Taux de complétion par colonne :")
for col in [c for c in df_owid_final.columns if c not in ['iso3', 'nom_owid', 'year']]:
    pct = df_owid_final[col].notna().mean() * 100
    print(f"  {col:<35} {pct:.0f}%")

📅 Lignes par année (5 ans d'intervalle) :
 year  nb_pays
 1965      164
 1970      164
 1975      164
 1980      164
 1985      164
 1990      165
 1995      184
 2000      186
 2005      186
 2010      192
 2015      192
 2020      200

📊 Taux de complétion par colonne :
  total_meat_kg_per_cap               100%
  total_meat_co2_per_cap              100%


In [30]:
# 7.3 — Résumé final
print("=" * 60)
print("✅ ENRICHISSEMENT OWID TERMINÉ")
print("=" * 60)
print(f"""
Fichier produit :
  📄 {OUTPUT_OWID.name}
     {len(df_owid_final):,} lignes
     {df_owid_final['iso3'].nunique()} pays
     {df_owid_final['year'].min()}–{df_owid_final['year'].max()}

Variables de consommation (kg/habitant/an) :
  • beef_kg_per_cap      (bœuf)
  • pork_kg_per_cap      (porc)
  • poultry_kg_per_cap   (volaille)
  • sheep_kg_per_cap     (mouton/agneau)
  • total_meat_kg_per_cap

Variables d'impact calculées (kgCO₂eq/habitant/an) :
  • beef_co2_per_cap     (× 59.6 kgCO₂/kg)
  • pork_co2_per_cap     (× 7.2  kgCO₂/kg)
  • poultry_co2_per_cap  (× 6.1  kgCO₂/kg)
  • sheep_co2_per_cap    (× 24.5 kgCO₂/kg)
  • total_meat_co2_per_cap

Utilisation dans le pipeline :
  → Features ML : corrélation conso viande ↔ impact total
  → Validation croisée avec production FAO
  → Jointure via iso3 dans 02_etl_pipeline.ipynb
""")

✅ ENRICHISSEMENT OWID TERMINÉ

Fichier produit :
  📄 owid_meat_consumption.csv
     10,949 lignes
     210 pays
     1961–2022

Variables de consommation (kg/habitant/an) :
  • beef_kg_per_cap      (bœuf)
  • pork_kg_per_cap      (porc)
  • poultry_kg_per_cap   (volaille)
  • sheep_kg_per_cap     (mouton/agneau)
  • total_meat_kg_per_cap

Variables d'impact calculées (kgCO₂eq/habitant/an) :
  • beef_co2_per_cap     (× 59.6 kgCO₂/kg)
  • pork_co2_per_cap     (× 7.2  kgCO₂/kg)
  • poultry_co2_per_cap  (× 6.1  kgCO₂/kg)
  • sheep_co2_per_cap    (× 24.5 kgCO₂/kg)
  • total_meat_co2_per_cap

Utilisation dans le pipeline :
  → Features ML : corrélation conso viande ↔ impact total
  → Validation croisée avec production FAO
  → Jointure via iso3 dans 02_etl_pipeline.ipynb

